# Task 14: Custom CUDA Kernel

## Objective

Implement a simple custom CUDA kernel and compare it with a standard
PyTorch operation.

This task demonstrates:

- CUDA
- GPU computation
- Custom CUDA kernels
- GPU memory
- Parallel computation
- Performance comparison

CUDA allows computations to be executed in parallel on NVIDIA GPUs.

In this task, we implement a simple vector addition operation.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Step 1: Check CUDA Availability

Before running a CUDA kernel, we need to check whether an NVIDIA GPU
and CUDA-enabled PyTorch installation are available.

In [ ]:
if torch.cuda.is_available():

    print("CUDA is available!")

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print("CUDA is NOT available.")
    print("A CUDA-capable NVIDIA GPU is required for this task.")

CUDA is available!
GPU: Tesla T4


In [ ]:
if not torch.cuda.is_available():

    print(
        "Skipping CUDA execution because no CUDA GPU is available."
    )

else:

    print(
        "Ready to execute CUDA operations."
    )

Ready to execute CUDA operations.


## Step 2: Create Data on the GPU

We create two large vectors and move them to the CUDA device.

The GPU can process many elements in parallel.

In [ ]:
if torch.cuda.is_available():

    size = 10_000_000

    a = torch.randn(
        size,
        device="cuda"
    )

    b = torch.randn(
        size,
        device="cuda"
    )

    print("Vector size:", size)
    print("Device:", a.device)

Vector size: 10000000
Device: cuda:0


## Step 3: Standard PyTorch GPU Operation

First, we perform vector addition using the built-in PyTorch operation.

This gives us a baseline for comparison.

In [ ]:
import time

if torch.cuda.is_available():

    torch.cuda.synchronize()

    start = time.perf_counter()

    c_torch = a + b

    torch.cuda.synchronize()

    torch_time = (
        time.perf_counter() - start
    )

    print(
        "PyTorch execution time:",
        round(torch_time, 6),
        "seconds"
    )

PyTorch execution time: 0.000793 seconds


## Step 4: Custom CUDA Kernel

A CUDA kernel is a function executed by many GPU threads in parallel.

Each thread handles one element of the vectors.

Conceptually:

Thread 0 → c[0] = a[0] + b[0]

Thread 1 → c[1] = a[1] + b[1]

Thread 2 → c[2] = a[2] + b[2]

...

This allows thousands of GPU threads to perform calculations
simultaneously.

## Step 5: Compile a Custom CUDA Extension

PyTorch can compile custom C++ and CUDA code and expose it as a
Python function.

The following kernel performs vector addition.

In [ ]:
if torch.cuda.is_available():

    from torch.utils.cpp_extension import load_inline

    cpp_source = r'''
    #include <torch/extension.h>

    torch::Tensor vector_add_cuda(
        torch::Tensor a,
        torch::Tensor b
    );

    PYBIND11_MODULE(
        TORCH_EXTENSION_NAME,
        m
    ) {
        m.def(
            "vector_add",
            &vector_add_cuda,
            "Custom CUDA Vector Addition"
        );
    }
    '''

    cuda_source = r'''
    #include <torch/extension.h>
    #include <cuda.h>
    #include <cuda_runtime.h>

    __global__ void vector_add_kernel(
        const float* a,
        const float* b,
        float* c,
        int n
    ) {

        int index =
            blockIdx.x * blockDim.x + threadIdx.x;

        if (index < n) {

            c[index] =
                a[index] + b[index];
        }
    }

    torch::Tensor vector_add_cuda(
        torch::Tensor a,
        torch::Tensor b
    ) {

        auto c =
            torch::zeros_like(a);

        int n = a.numel();

        const int threads = 256;

        const int blocks =
            (n + threads - 1) / threads;

        vector_add_kernel<<<
            blocks,
            threads
        >>>(
            a.data_ptr<float>(),
            b.data_ptr<float>(),
            c.data_ptr<float>(),
            n
        );

        return c;
    }
    '''

    print("CUDA source prepared.")

CUDA source prepared.


## Step 6: Compile the CUDA Kernel

The CUDA source is compiled into a PyTorch extension.

Compilation may take some time the first time it is executed.

In [ ]:
!pip install ninja

import shutil
import os

if torch.cuda.is_available():
    # Define the cache directory
    cache_dir = torch.utils.cpp_extension._get_build_directory("custom_vector_add", verbose=True)

    # Check if the cache directory exists and remove it
    if os.path.exists(cache_dir):
        print(f"Removing old cache directory: {cache_dir}")
        shutil.rmtree(cache_dir)
        print("Cache removed successfully.")
    else:
        print(f"Cache directory not found: {cache_dir}. No cleanup needed.")

    # Define an explicit build directory to try and debug silent compilation failures
    explicit_build_dir = "/tmp/custom_cuda_build"
    if os.path.exists(explicit_build_dir):
        print(f"Removing old explicit build directory: {explicit_build_dir}")
        shutil.rmtree(explicit_build_dir)
        print("Explicit build directory removed successfully.")

    # Create the explicit build directory if it doesn't exist
    os.makedirs(explicit_build_dir, exist_ok=True)
    print(f"Ensured explicit build directory exists: {explicit_build_dir}")

    cuda_module = load_inline(
        name="custom_vector_add",
        cpp_sources=cpp_source,
        cuda_sources=cuda_source,
        functions=None,
        extra_cflags=["-O3"],
        extra_cuda_cflags=["-O3"],
        build_directory=explicit_build_dir,
        verbose=True
    )

    print(
        "Custom CUDA kernel compiled successfully!"
    )

Removing old cache directory: /root/.cache/torch_extensions/py312_cu128/custom_vector_add
Cache removed successfully.
Removing old explicit build directory: /tmp/custom_cuda_build
Explicit build directory removed successfully.
Ensured explicit build directory exists: /tmp/custom_cuda_build
Custom CUDA kernel compiled successfully!


## Step 7: Execute the Custom CUDA Kernel

Now we use our own CUDA kernel instead of PyTorch's built-in addition.

In [ ]:
if torch.cuda.is_available():

    torch.cuda.synchronize()

    start = time.perf_counter()

    c_cuda = cuda_module.vector_add(
        a,
        b
    )

    torch.cuda.synchronize()

    cuda_time = (
        time.perf_counter() - start
    )

    print(
        "Custom CUDA execution time:",
        round(cuda_time, 6),
        "seconds"
    )

Custom CUDA execution time: 0.013063 seconds


## Step 8: Verify the Results

The custom CUDA kernel should produce the same result as the standard
PyTorch operation.

In [ ]:
if torch.cuda.is_available():

    is_correct = torch.allclose(
        c_torch,
        c_cuda,
        atol=1e-6
    )

    print(
        "Results match:",
        is_correct
    )

Results match: True


## Step 9: Performance Comparison

We compare the execution time of:

1. Standard PyTorch GPU addition
2. Custom CUDA vector addition

The custom kernel demonstrates how GPU operations can be implemented
directly using CUDA.

In [ ]:
if torch.cuda.is_available():

    print(
        "PyTorch time:",
        round(torch_time, 6),
        "seconds"
    )

    print(
        "Custom CUDA time:",
        round(cuda_time, 6),
        "seconds"
    )

    if cuda_time < torch_time:

        print(
            "Custom CUDA kernel was faster."
        )

    else:

        print(
            "PyTorch operation was faster."
        )

PyTorch time: 0.000793 seconds
Custom CUDA time: 0.013063 seconds
PyTorch operation was faster.


## Step 10: CUDA Threads and Blocks

CUDA organizes GPU execution using:

- Threads
- Blocks
- Grids

In our kernel:

threads per block = 256

The number of blocks is calculated as:

blocks = ceil(number_of_elements / threads_per_block)

Each thread processes one vector element.

In [ ]:
if torch.cuda.is_available():

    threads = 256

    blocks = (
        size + threads - 1
    ) // threads

    print(
        "Threads per block:",
        threads
    )

    print(
        "Number of blocks:",
        blocks
    )

    print(
        "Total elements:",
        size
    )

Threads per block: 256
Number of blocks: 39063
Total elements: 10000000


## Step 11: Small Example

We verify the custom CUDA operation using a small vector that is easy
to understand.

In [ ]:
if torch.cuda.is_available():

    x = torch.tensor(
        [1, 2, 3, 4, 5],
        dtype=torch.float32,
        device="cuda"
    )

    y = torch.tensor(
        [10, 20, 30, 40, 50],
        dtype=torch.float32,
        device="cuda"
    )

    result = cuda_module.vector_add(
        x,
        y
    )

    print(
        "x:",
        x.cpu()
    )

    print(
        "y:",
        y.cpu()
    )

    print(
        "x + y:",
        result.cpu()
    )

x: tensor([1., 2., 3., 4., 5.])
y: tensor([10., 20., 30., 40., 50.])
x + y: tensor([11., 22., 33., 44., 55.])


## Results

The custom CUDA experiment demonstrates:

- CUDA availability checking
- GPU tensor computation
- CUDA kernels
- Threads and blocks
- Parallel vector addition
- Custom CUDA extension
- Performance measurement
- Result verification

In [ ]:
if torch.cuda.is_available():

    print("========== TASK 14 RESULTS ==========")

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "Vector size:",
        size
    )

    print(
        "Threads per block:",
        256
    )

    print(
        "CUDA blocks:",
        blocks
    )

    print(
        "Results match:",
        torch.allclose(
            c_torch,
            c_cuda
        )
    )

    print("=====================================")

else:

    print("========== TASK 14 ==========")

    print(
        "CUDA GPU not available."
    )

    print(
        "Run this notebook on a CUDA-enabled GPU"
    )

    print(
        "to execute the custom kernel."
    )

    print("=============================")

========== TASK 14 RESULTS ==========
GPU: Tesla T4
Vector size: 10000000
Threads per block: 256
CUDA blocks: 39063
Results match: True


# Conclusion

A custom CUDA vector addition kernel was implemented using PyTorch's
CUDA extension mechanism.

The kernel assigns individual vector elements to GPU threads, allowing
the addition operation to be performed in parallel.

The results were compared with the standard PyTorch implementation to
verify correctness and observe execution performance.

This task provides a basic understanding of how custom CUDA kernels
can accelerate computational workloads using GPU parallelism.